In [1]:
import sys
sys.path.append('../Common')

import CommonBinance

In [2]:
def detectSignal(symbol, from_date, to_date, timeframe):

    import pandas as pd
    import plotly.graph_objects as go
    import redis
    import numpy as np
    from datetime import datetime
    from statsmodels.tsa.arima.model import ARIMA

    # ##############################################Step 1: Lấy dữ liệu##############################################
    data = CommonBinance.CommonBinance.loaddataBinance_FromTo(symbol, from_date, to_date, timeframe)
    
    # ##############################################Step 2: Chiến lược##############################################  
    # Giả sử `data` là DataFrame chứa các cột Datetime, Open, High, Low, Close, Volume
    # Đặt chỉ mục DataFrame với cột 'Datetime' có dữ liệu kiểu datetime
    data.set_index('Datetime', inplace=True)

    # Tính Momentum
    data['Momentum'] = data['Close'].diff()

    # Tính RSI
    delta = data['Close'].diff()
    up, down = delta.clip(lower=0), delta.clip(upper=0).abs()
    roll_up, roll_down = up.rolling(14).mean(), down.rolling(14).mean()
    RS = roll_up / roll_down
    data['RSI'] = 100.0 - (100.0 / (1.0 + RS))

    # Chọn tham số ARIMA dựa trên AIC, BIC hoặc các tiêu chí khác (cần phải thực hiện trước) => Chay cell o tren
    model = ARIMA(data['Close'], order=(2, 1, 2))
    model_fit = model.fit()

    # Dự đoán giá
    data['Predicted_Close'] = model_fit.predict(start=0, end=len(data)-1, typ='levels')

    # Xác định tín hiệu mua và bán
    data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['Momentum'] > -10) & (data['RSI'] < 170))
    data['Sell_Signal'] = ((data['Predicted_Close'] < data['Close']) & (data['Momentum'] < 2000) & (data['RSI'] > 30))

    # ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
    # Nếu có tín hiệu thì đẩy qua Redis
    # Datetime  Open    High    Low	Close   Volume  SMA short_ema   long_ema    MACD    Signal_Line Buy_Signal  Sell_Signal
    # Tạo kết nối Redis
    r = redis.Redis(host='localhost', port=6379, db=0)
    # Tạo hash key
    hash_key = 'OG_CRTOFT_Arima, Momentum, RSI'
    last_record = data.iloc[-1]
    print(last_record)
    # Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
    if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
        for field, value in last_record.to_dict().items():
            # Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
            if isinstance(value, pd.Timestamp):
                value = value.isoformat()
            elif isinstance(value, (int, np.uint64)):
                value = str(value)
            r.hset(hash_key, field, value)  
            r.hset(hash_key, 'Symbol', symbol)
            r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
        print(last_record)   
    else:
        print('Không có tín hiệu!')   

In [3]:
from datetime import datetime, timedelta
import schedule
import time

def schedule_detectSignal():
    symbol = 'ETHUSDT'
    from_date = (datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
    to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
    timeframe = '1m' # 1m
    detectSignal(symbol, from_date, to_date, timeframe)

# Task chay: Viet theo While True
# Danh sách các phút cụ thể bạn muốn hàm được chạy
# run_minutes = [17, 27, 50, 2]
run_minutes = list(range(0,60,1))

while True:
    current_time = datetime.now()
    current_minute = current_time.minute
    # Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
    if current_minute in run_minutes:
        # Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
        last_run = getattr(schedule_detectSignal, 'last_run', None)
        if last_run is None or last_run != current_minute:
            schedule_detectSignal()
            # Lưu lại phút cuối cùng mà hàm đã chạy
            setattr(schedule_detectSignal, 'last_run', current_minute)   

    # Nghỉ 1 giây trước khi kiểm tra lại
    time.sleep(1)

# Task chay viet theo Schedule
# Danh sách các phút mà bạn muốn chạy hàm
# run_minutes = [0, 23, 33, 35]
# def job():
#     current_minute = datetime.now().minute
#     if current_minute in run_minutes:
#         schedule_detectSignal()
# # Thiết lập lịch trình gọi hàm job mỗi phút
# schedule.every().minute.do(job)
# while True:
#     schedule.run_pending()
#     time.sleep(1)  # Kiểm tra lịch trình mỗi 1 giây thay vì 10 giây để tăng độ chính xác

C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)


Open                   3636.07
High                   3647.98
Low                    3636.07
Close                  3642.44
Volume               2058.7065
Momentum                  6.38
RSI                  71.228733
Predicted_Close    3636.532999
Buy_Signal               False
Sell_Signal               True
Name: 2024-11-29 15:14:00, dtype: object
Open                   3636.07
High                   3647.98
Low                    3636.07
Close                  3642.44
Volume               2058.7065
Momentum                  6.38
RSI                  71.228733
Predicted_Close    3636.532999
Buy_Signal               False
Sell_Signal               True
Name: 2024-11-29 15:14:00, dtype: object


C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\Lo

Open                   3642.44
High                   3643.02
Low                    3641.97
Close                  3642.67
Volume                 128.763
Momentum                  0.22
RSI                   73.80031
Predicted_Close    3642.005722
Buy_Signal               False
Sell_Signal               True
Name: 2024-11-29 15:15:00, dtype: object
Open                   3642.44
High                   3643.02
Low                    3641.97
Close                  3642.67
Volume                 128.763
Momentum                  0.22
RSI                   73.80031
Predicted_Close    3642.005722
Buy_Signal               False
Sell_Signal               True
Name: 2024-11-29 15:15:00, dtype: object


C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\Lo

...

Open                 3642.83
High                 3643.26
Low                  3642.62
Close                3642.63
Volume                35.293
Momentum                -0.2
RSI                72.105263
Predicted_Close    3642.7108
Buy_Signal              True
Sell_Signal            False
Name: 2024-11-29 15:20:00, dtype: object
Open                 3642.83
High                 3643.26
Low                  3642.62
Close                3642.63
Volume                35.293
Momentum                -0.2
RSI                72.105263
Predicted_Close    3642.7108
Buy_Signal              True
Sell_Signal            False
Name: 2024-11-29 15:20:00, dtype: object


C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)


Open                   3639.59
High                   3639.59
Low                    3638.16
Close                  3638.83
Volume                138.5324
Momentum                 -0.75
RSI                  62.820513
Predicted_Close    3639.196073
Buy_Signal                True
Sell_Signal              False
Name: 2024-11-29 15:21:00, dtype: object
Open                   3639.59
High                   3639.59
Low                    3638.16
Close                  3638.83
Volume                138.5324
Momentum                 -0.75
RSI                  62.820513
Predicted_Close    3639.196073
Buy_Signal                True
Sell_Signal              False
Name: 2024-11-29 15:21:00, dtype: object


C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)
C:\Users\dongn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency min will be used.
  self._init_dates(dates, freq)


LinAlgError: LU decomposition error.